# 01 – Explore the Video_Games 5-core splits
Goal: check that the files match the dataset page, that the time split is clean,
and how much of valid/test is new relative to training (cold start).

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth",None) # to see full history strings

DATA_DIR = Path("../data/raw")

T1 = pd.to_datetime(1628643414042, unit="ms", utc=True)  # train | valid cutoff (dataset page)
T2 = pd.to_datetime(1658002729837, unit="ms", utc=True)  # valid | test cutoff (dataset page)


splits = {
    name : pd.read_csv(DATA_DIR/ f"Video_Games.{name}.csv") for name in ("train","valid","test")
}

for df in splits.values():
    df["timestamp"] = pd.to_datetime(df["timestamp"],unit="ms",utc=True)
train,valid,test = splits["train"],splits["valid"],splits["test"]

## 1. Shape, dtypes, time ranges, missing history
Question: do the files match the dataset page, and do the splits sit on the right side of the cutoffs?

In [2]:
print(train.dtypes,"\n")

summary = pd.DataFrame({
    name :{
        "rows" : len(df),
        "first" : df["timestamp"].min(),
        "last" : df["timestamp"].max(),
        "missing_history" : df["history"].isna().sum(),
        "missing_history_%" : round(df["history"].isna().mean()*100,1),
    }
    for name,df in splits.items()
}).T

# splits must sit on the right side of the cutoffs (errors out if not)
assert train["timestamp"].max() < T1 <= valid["timestamp"].min()
assert valid["timestamp"].max() < T2 <= test["timestamp"].min()
summary


user_id                        str
parent_asin                    str
rating                     float64
timestamp      datetime64[us, UTC]
history                        str
dtype: object 



,rows,first,last,missing_history,missing_history_%
train,736827,1999-10-22 21:43:15+00:00,2021-08-11 00:52:24.373000+00:00,91562,12.4
valid,34510,2021-08-11 01:18:44.721000+00:00,2022-07-16 20:10:37.581000+00:00,1416,4.1
test,43249,2022-07-16 20:27:28.622000+00:00,2023-09-02 18:14:28.630000+00:00,1784,4.1


## 2. Cold start: users and games that are new in valid/test
Question: how many users and games never appear in training, and how many rows do they affect?

In [3]:
def unseen_report(df,reference,col):
    """Unique values of `col` in `df` that never appear in `reference`,
    and the share of `df` rows they affect."""

    seen = df[col].isin(reference[col])
    return {
        "unique_total" : df[col].nunique(),
        "unique_unseen" : df.loc[~seen,col].nunique(),
        "rows_unseen_%" : round((~seen).mean()*100,2),
    }

train_valid = pd.concat([train,valid])

rows = {}
for split_name,df in [("valid",valid),("test",test)]:
    rows[(split_name,"users","vs train")] = unseen_report(df,train,"user_id")
    rows[(split_name,"games","vs train")] = unseen_report(df,train,"parent_asin")

rows[("test", "users", "vs train+valid")] = unseen_report(test, train_valid, "user_id")
rows[("test", "games", "vs train+valid")] = unseen_report(test, train_valid, "parent_asin")

pd.DataFrame(rows).T

unique_total  unique_unseen  rows_unseen_%
valid users vs train             15655.0         1416.0          18.56
      games vs train              7620.0         1094.0          27.86
test  users vs train             13963.0         2680.0          41.01
      games vs train              7671.0         2543.0          60.48
      users vs train+valid       13963.0         1784.0          32.28
      games vs train+valid        7671.0         1542.0          42.96

## 3. Ratings
Question: how are ratings distributed, and how much data is left if only 4–5 stars count as "liked"?

In [4]:
train["rating"].value_counts(normalize=True) , test["rating"].value_counts(normalize=True)

(rating
 5.0    0.637644
 4.0    0.166614
 3.0    0.087058
 1.0    0.063514
 2.0    0.045169
 Name: proportion, dtype: float64,
 rating
 5.0    0.654027
 4.0    0.168235
 3.0    0.076395
 1.0    0.060533
 2.0    0.040810
 Name: proportion, dtype: float64)

## 4. How `history` is built
Question: what does `history` contain for consecutive rows of one user?
Result: each row's `history` is the user's earlier games, so it grows by one game per row. The first review has no history (NaN).

In [5]:
train.head(10)

,user_id,parent_asin,rating,timestamp,history
0,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08R5B7YS4,1.0,2021-01-24 03:41:06.223+00:00,NaN
1,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B0863MT183,4.0,2021-02-19 02:33:06.538+00:00,B08R5B7YS4
2,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08P8P7686,5.0,2021-02-19 02:35:12.995+00:00,B08R5B7YS4 B0863MT183
3,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B0B7LV3DN2,4.0,2021-04-05 16:50:45.475+00:00,B08R5B7YS4 B0863MT183 B08P8P7686
4,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B09WMQ6DXG,5.0,2021-05-05 16:16:08.468+00:00,B08R5B7YS4 B0863MT183 B08P8P7686 B0B7LV3DN2
5,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B00KTORA0K,5.0,2014-12-12 15:07:53.000+00:00,NaN
6,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B0792QJ28Y,5.0,2021-01-07 17:26:26.669+00:00,B00KTORA0K
7,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B07VGRJDFY,5.0,2021-01-28 05:17:34.257+00:00,B00KTORA0K B0792QJ28Y
8,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B07J6ZZ83K,3.0,2021-01-28 05:27:48.884+00:00,B00KTORA0K B0792QJ28Y B07VGRJDFY
9,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B0C3KYVDWT,5.0,2021-01-28 05:31:36.446+00:00,B00KTORA0K B0792QJ28Y B07VGRJDFY B07J6ZZ83K


## 5. Activity per user in each split
Question: how many rows does a typical user have in train, valid and test, and how heavy are the heaviest users?

In [6]:
for name, df in splits.items():
    print(name)
    print(df.groupby("user_id").size().describe(), "\n")

train
count    91562.000000
mean         8.047301
std          7.741370
min          1.000000
25%          5.000000
50%          6.000000
75%          9.000000
max        473.000000
dtype: float64 

valid
count    15655.000000
mean         2.204408
std          2.296501
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max         50.000000
dtype: float64 

test
count    13963.000000
mean         3.097400
std          4.042281
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max        184.000000
dtype: float64 



## 6. 5-core, popularity and duplicate checks
Questions: does the 5-core rule hold across all splits? How concentrated are reviews on a few games? Are there duplicate user–game pairs?

In [7]:
all_rows = pd.concat([train,valid,test],ignore_index=True)

# 1. 5-core check for users: every user should have at least 5 rows overall
rows_per_user = all_rows.groupby("user_id").size()
print("rows per user (all splits):")
print(rows_per_user.describe(),"\n")
print("users with fewer than 5 rows:", (rows_per_user < 5).sum(),"\n")

# 2. Game popularity (also checks the 5-core rule for games)
rows_per_game = all_rows.groupby("parent_asin").size().sort_values(ascending=False)
print("rows per game (all splits):")
print(rows_per_game.describe(),"\n")
print("games with fewer than 5 rows:", (rows_per_game < 5).sum(),"\n")

top_1pct = int(len(rows_per_game)*0.01)
share = rows_per_game.head(top_1pct).sum() / len(all_rows)
print(f"top 1% of games ({top_1pct} games) account for {share:.1%} of all rows\n")

# 3. Duplicates: the same user reviewing the same game more than once
print("duplicate (user, game) pairs:", all_rows.duplicated(subset=["user_id", "parent_asin"]).sum())

rows per user (all splits):
count    94762.000000
mean         8.596125
std          7.898208
min          5.000000
25%          5.000000
50%          6.000000
75%          9.000000
max        473.000000
dtype: float64 

users with fewer than 5 rows: 0 

rows per game (all splits):
count    25612.000000
mean        31.804857
std         78.046151
min          5.000000
25%          7.000000
50%         12.000000
75%         26.000000
max       2955.000000
dtype: float64 

games with fewer than 5 rows: 0 

top 1% of games (256 games) account for 18.6% of all rows

duplicate (user, game) pairs: 0


## Findings
- Files match the dataset page: 736,827 / 34,510 / 43,249 rows (train / valid / test). The cutoffs split cleanly: train before 2021-08-11, valid until 2022-07-16, test until 2023-09-02.
- 94,762 users, 25,612 games, ~815K interactions. No duplicate user–game pairs; every user and game has at least 5 rows.
- A missing `history` marks a user's first review (12.4% of train rows, ~4% of valid and test).
- Users: median 6 rows, mean 8.6, max 473. Most users sit near the 5-review floor, with a few heavy users. In valid/test most users have 1–2 rows; the heaviest has 184.
- Games: median 12 reviews, mean 31.8, max 2,955. The top 1% of games (256) hold 18.6% of all rows.
- Ratings (train): 64% five-star, 17% four-star, 20% three-star or lower. Test is similar (65% / 17% / 18%).
- Cold-start users: 1,416 valid and 2,680 test users never appear in train. By rows, 41% of test rows (32% against train+valid) come from users with no earlier training data.
- Cold-start games: 27.9% of valid rows and 60.5% of test rows involve a game not in train. Even with valid added, 43.0% of test rows (1,542 games) do. An ID-only model can hit at most ~57% of test rows.

## Next steps
1. Check whether item metadata covers all games, especially the 1,542 unseen ones (`02_explore_metadata.ipynb`).
2. Define the evaluation setup (per-row vs per-user, rows with no history, seen vs new games).
3. Build the baselines (most popular, co-occurrence).